# Week 6 — Data Representation & Model Evaluation

**Part 1** covers representing data well (encoding, feature engineering,
missing values, outliers, leakage-safe scaling). **Part 2** covers
evaluating a model honestly once you have it (cross-validation, confusion
matrix, precision/recall, ROC/AUC) — the natural next question once your
data is in good shape.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

## 1. Build a synthetic used-car dataset

We generate 300 used car listings with a known (but noisy) pricing rule, so we
can see clearly how much each data-representation trick helps.

In [ ]:
rng = np.random.default_rng(7)
n = 300

brands = rng.choice(["Hyundai", "Kia", "Toyota", "BMW"], size=n, p=[0.35, 0.3, 0.2, 0.15])
fuel_types = rng.choice(["gasoline", "diesel", "electric"], size=n, p=[0.55, 0.3, 0.15])
transmissions = rng.choice(["automatic", "manual"], size=n, p=[0.8, 0.2])
purchase_year = rng.integers(2010, 2024, size=n)
mileage_km = rng.integers(5_000, 180_000, size=n)

brand_premium = {"Hyundai": 0, "Kia": 0, "Toyota": 3_000, "BMW": 12_000}
fuel_premium = {"gasoline": 0, "diesel": 1_000, "electric": 6_000}

age = 2024 - purchase_year
base_price = 28_000
price = (
    base_price
    - age * 1_400                 # older car -> cheaper
    - mileage_km * 0.05           # more mileage -> cheaper
    + np.array([brand_premium[b] for b in brands])
    + np.array([fuel_premium[f] for f in fuel_types])
    + rng.normal(0, 1_500, size=n)  # noise: negotiation, condition, luck
)
price = np.clip(price, 2_000, None)

cars = pd.DataFrame({
    "brand": brands,
    "fuel_type": fuel_types,
    "transmission": transmissions,
    "purchase_year": purchase_year,
    "mileage_km": mileage_km,
    "price": price.round(0),
})
cars.head()

## Step 1: Numbers-only baseline (ignore the categories)

The naive approach: drop every text column and use only the numeric ones.

In [ ]:
def evaluate(X, y, label):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)
    model = LinearRegression().fit(Xtr, ytr)
    mae = mean_absolute_error(yte, model.predict(Xte))
    print(f"{label:35s} MAE = ${mae:,.0f}")
    return mae

results = {}
X1 = cars[["purchase_year", "mileage_km"]]
results["1. numeric only"] = evaluate(X1, cars["price"], "1. Numeric columns only")

## Step 2: One-hot encode the categorical columns

`pd.get_dummies` turns each category into its own 0/1 column, so "brand" (a
word) becomes 4 numeric columns the model can use — without implying any order
between brands.

In [ ]:
categorical_encoded = pd.get_dummies(cars[["brand", "fuel_type", "transmission"]])
categorical_encoded.head()

In [ ]:
X2 = pd.concat([cars[["purchase_year", "mileage_km"]], categorical_encoded], axis=1)
results["2. + one-hot categories"] = evaluate(X2, cars["price"], "2. + one-hot encoded categories")

## Step 3: Feature engineering — `age` instead of `purchase_year`

A linear model can only learn "the *bigger* this number, the *more/less* the
price changes." `purchase_year` (e.g. 2010 vs. 2023) works, but `age` (car's
age in years) is a more natural, directly-meaningful feature for the same
underlying relationship. Feature engineering means *reshaping* raw values into
a form that's easier for the model to use.

In [ ]:
cars["age"] = 2024 - cars["purchase_year"]

X3 = pd.concat([cars[["age", "mileage_km"]], categorical_encoded], axis=1)
results["3. + engineered 'age' feature"] = evaluate(X3, cars["price"], "3. + engineered 'age' feature")

## Step 4: Scale the numeric features

`mileage_km` ranges up to 180,000 while `age` maxes out around 14 — very
different scales. Some models (and especially neural networks, coming in
Week 8) train better when every feature is standardized to a similar range.

In [ ]:
numeric_cols = ["age", "mileage_km"]
scaler = StandardScaler()
X4 = X3.copy()
X4[numeric_cols] = scaler.fit_transform(X4[numeric_cols])

results["4. + scaled numeric features"] = evaluate(X4, cars["price"], "4. + scaled numeric features")

## Compare all four representations

In [ ]:
summary = pd.Series(results)
print(summary)

plt.figure(figsize=(7, 4))
summary.plot(kind="barh", color="steelblue")
plt.title("Prediction Error (MAE, lower = better) by Data Representation")
plt.xlabel("Mean Absolute Error ($)")
plt.gca().invert_yaxis()
plt.show()

## 5. Real-world practice: Titanic — missing values, outliers, leakage-safe scaling

So far we've used a synthetic used-car dataset to practice encoding and
feature engineering. Now let's use real
[Titanic](https://www.kaggle.com/datasets/yasserh/titanic-dataset) passenger
data to practice the rest of this week's preprocessing topics — **missing
values**, **outliers**, and **avoiding data leakage when scaling**. This is a
plain dataset (not a Kaggle competition), so it downloads with no login
required.

In [ ]:
!pip install -q kagglehub
import kagglehub
import glob

titanic_dir = kagglehub.dataset_download("yasserh/titanic-dataset")
csv_path = glob.glob(f"{titanic_dir}/**/*.csv", recursive=True)[0]
print("Using file:", csv_path)

titanic = pd.read_csv(csv_path)
print("Shape:", titanic.shape)
titanic.head()

### Checking for missing values

As covered in lecture, missing values don't mean we should just drop every
row that has one — real-world data has missing values all the time, and
dropping too aggressively throws away good data. First let's see which
columns have how much missing.

In [ ]:
print(titanic.isna().sum())
print(f"\nAge missing: {titanic['Age'].isna().mean():.1%}")
print(f"Cabin missing: {titanic['Cabin'].isna().mean():.1%}")
print(f"Embarked missing: {titanic['Embarked'].isna().mean():.1%}")

### Three different strategies for three different columns

Matching the strategies from lecture to what each column actually needs:

- **`Cabin`**: over 70% missing — at this rate, imputing is unreliable, so
  we **drop the whole column** instead.
- **`Embarked`**: only 2 missing values — fill with the **mode** (most
  common value).
- **`Age`**: about 20% missing — too much to drop, but a single global mean
  would be crude. We use the lecture's "mean within the same category" idea:
  group by `Pclass` and `Sex`, and fill with **that group's median** (median
  is less sensitive to extreme values than mean).

In [ ]:
titanic = titanic.drop(columns=["Cabin"])
print("Dropped Cabin (over 70% missing)")

embarked_mode = titanic["Embarked"].mode()[0]
titanic["Embarked"] = titanic["Embarked"].fillna(embarked_mode)
print(f"Filled Embarked missing values with the mode: '{embarked_mode}'")

titanic["Age"] = titanic.groupby(["Pclass", "Sex"])["Age"].transform(lambda s: s.fillna(s.median()))
titanic["Age"] = titanic["Age"].fillna(titanic["Age"].median())  # fallback if a group has no valid values at all
print("Filled Age missing values with the (Pclass, Sex) group median")

print("\nMissing values after cleaning:")
print(titanic.isna().sum())

### A quick example: multiple imputation (MICE) on real ratings

The three Titanic fixes above were deliberately simple — `Age` only really
correlates with `Pclass`/`Sex`, so a group median already captures most of
what a fancier method could add. Lecture also introduced a more powerful
option: **multiple imputation (MICE)**, which fills every missing cell by
iteratively regressing each column on all the others. Here it is applied to
something with much richer structure — the **user x movie rating matrix**
from Week 5's clustering capstone (real [MovieLens](https://movielens.org/)
data).

This is just a demonstration of *how* to use MICE on real data, not a
comparison — judging whether MICE actually did a *good* job here is hard,
since there's no ground truth for a rating nobody ever gave. For an actual
head-to-head comparison between imputation methods, the next section uses a
dataset where we *do* have ground truth to check against.

In [ ]:
import urllib.request

MOVIELENS_BASE_URL = "https://raw.githubusercontent.com/digantamisra98/KMeans-Movie-Ratings/master"
urllib.request.urlretrieve(f"{MOVIELENS_BASE_URL}/movies.csv", "ml_movies.csv")
urllib.request.urlretrieve(f"{MOVIELENS_BASE_URL}/ratings.csv", "ml_ratings.csv")

ml_ratings = pd.read_csv("ml_ratings.csv")
ml_movies = pd.read_csv("ml_movies.csv")
print("ratings:", ml_ratings.shape, " movies:", ml_movies.shape)
print("users:", ml_ratings["userId"].nunique(), " movies rated:", ml_ratings["movieId"].nunique())

### Rebuilding Week 5's user x reference-movie matrix

Same idea as Week 5: pick a genre-balanced set of popular "reference
movies," and represent each eligible user as their rating vector over just
those movies — but this time keep the real blanks instead of immediately
filling them with each user's own average.

In [ ]:
genre_targets = {
    "Action": 5, "Comedy": 5, "Drama": 5, "Sci-Fi": 5,
    "Animation": 3, "Romance": 3, "Horror": 2,
}
POPULAR_MIN = 50

rating_counts = ml_ratings.groupby("movieId").size().rename("n_ratings")
ml_movies_stats = ml_movies.merge(rating_counts, on="movieId")
ml_popular = ml_movies_stats[ml_movies_stats["n_ratings"] >= POPULAR_MIN]

reference_ids = []
for genre, n in genre_targets.items():
    candidates = ml_popular[ml_popular["genres"].str.contains(genre)].sort_values("n_ratings", ascending=False)
    reference_ids.extend(candidates["movieId"].head(n).tolist())
reference_ids = list(dict.fromkeys(reference_ids))
reference_movies = ml_movies_stats[ml_movies_stats["movieId"].isin(reference_ids)].reset_index(drop=True)

ELIGIBLE_FRAC = 0.5
ref_ratings = ml_ratings[ml_ratings["movieId"].isin(reference_movies["movieId"])]
n_ref = len(reference_movies)
user_ref_counts = ref_ratings.groupby("userId")["movieId"].nunique()
eligible_users = user_ref_counts[user_ref_counts >= ELIGIBLE_FRAC * n_ref].index

movie_pivot = ref_ratings[ref_ratings["userId"].isin(eligible_users)].pivot_table(
    index="userId", columns="movieId", values="rating"
)
movie_pivot = movie_pivot.reindex(columns=reference_movies["movieId"].values)

n_missing = movie_pivot.isna().sum().sum()
n_observed = movie_pivot.notna().sum().sum()
print(f"matrix shape: {movie_pivot.shape}  (users x reference movies)")
print(f"observed ratings: {n_observed}   missing: {n_missing}  ({n_missing / (n_missing + n_observed):.1%})")

### Filling it in with MICE

`IterativeImputer` treats every blank in the matrix as a regression target
predicted from everything else a user rated, and refines all of them
together over several passes.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401 -- unlocks IterativeImputer
from sklearn.impute import IterativeImputer

print("missing before:", movie_pivot.isna().sum().sum())

mice_filled_matrix = pd.DataFrame(
    IterativeImputer(random_state=0, max_iter=50, tol=1e-2).fit_transform(movie_pivot.values),
    index=movie_pivot.index, columns=movie_pivot.columns,
)
print("missing after:", mice_filled_matrix.isna().sum().sum())

# Compare a few of MICE's guesses against the flat per-user-mean fill Week 5 used
example_user = movie_pivot.isna().sum(axis=1).idxmax()  # the user with the most blanks
user_mean_fill = movie_pivot.loc[example_user].fillna(movie_pivot.loc[example_user].mean())
was_missing = movie_pivot.loc[example_user].isna()
comparison = pd.DataFrame({
    "MICE fill": mice_filled_matrix.loc[example_user],
    "flat user-mean fill": user_mean_fill,
})[was_missing]
print(f"\nUser {example_user}'s filled-in ratings, MICE vs. a flat per-user mean:")
print(comparison.round(2))

### A rigorous comparison: Wine Quality + Random Forest

To actually judge which missing-value method is "better," we need a
dataset with a real, known target to predict — then we can measure the one
thing that actually matters: **does the downstream prediction get worse**
when we rely on an imputed value instead of the real one? This section
runs the comparison from lecture:

1. Train a Random Forest on the complete, original data and measure its
   baseline performance.
2. Randomly knock out some of the input features (turn real values into
   missing ones) at an increasing rate: **5%, 10%, 20%, 30%**.
3. Fill the blanks back in with each of mean / median / mode / regression /
   MICE.
4. Feed each filled version through the *same, already-trained* Random
   Forest and see how far its predictions drift from the baseline.

Dataset: the classic [UCI Wine Quality (red)](https://archive.ics.uci.edu/dataset/186/wine+quality)
dataset — 1,599 wines, 11 physicochemical measurements (acidity, sugar,
alcohol, etc.), predicting a 0-10 quality score.

In [ ]:
WINE_COLUMNS = [
    "fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar", "chlorides",
    "free_sulfur_dioxide", "total_sulfur_dioxide", "density", "pH", "sulphates", "alcohol", "quality",
]
wine = pd.read_csv(
    "https://raw.githubusercontent.com/jbrownlee/Datasets/master/winequality-red.csv",
    names=WINE_COLUMNS,
)
print("Shape:", wine.shape)
wine.head()

### Step 1: Baseline — train and evaluate on the complete data

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

wine_feature_cols = [c for c in WINE_COLUMNS if c != "quality"]
X_wine = wine[wine_feature_cols]
y_wine = wine["quality"]

wine_X_train, wine_X_test, wine_y_train, wine_y_test = train_test_split(
    X_wine, y_wine, test_size=0.25, random_state=42
)

wine_rf = RandomForestRegressor(n_estimators=200, random_state=42)
wine_rf.fit(wine_X_train, wine_y_train)

baseline_rmse = np.sqrt(mean_squared_error(wine_y_test, wine_rf.predict(wine_X_test)))
print(f"Baseline RMSE (no missing values at all): {baseline_rmse:.4f}")

### Steps 2-4: knock out features, then fill them back in five different ways

`introduce_missing` turns a random fraction of a feature matrix's cells
into `NaN` (missing completely at random). `impute_test_features` then
fills those blanks back in five ways — each one fit only on the clean
training data, exactly as it would be in a real pipeline where you don't
get to peek at the real values you're trying to recover.

In [ ]:
def introduce_missing(df, ratio, rng):
    values = df.values.copy()
    n_missing = int(ratio * values.size)
    flat_positions = rng.choice(values.size, size=n_missing, replace=False)
    rows, cols = np.unravel_index(flat_positions, values.shape)
    values[rows, cols] = np.nan
    return pd.DataFrame(values, columns=df.columns, index=df.index)

def impute_test_features(X_train_clean, X_test_missing):
    filled = {}
    train_mean = X_train_clean.mean()
    filled["mean"] = X_test_missing.fillna(train_mean)
    filled["median"] = X_test_missing.fillna(X_train_clean.median())
    filled["mode"] = X_test_missing.fillna(X_train_clean.mode().iloc[0])

    # Regression: fit on the clean training rows only, then predict each test
    # row's missing feature from its OTHER (mean-filled) features.
    test_mean_filled = filled["mean"].values
    regression_filled = X_test_missing.values.copy()
    for j, col in enumerate(X_test_missing.columns):
        is_missing = X_test_missing[col].isna().values
        if not is_missing.any():
            continue
        other_cols = [k for k in range(X_train_clean.shape[1]) if k != j]
        model = LinearRegression().fit(X_train_clean.values[:, other_cols], X_train_clean[col].values)
        regression_filled[is_missing, j] = model.predict(test_mean_filled[is_missing][:, other_cols])
    filled["regression"] = pd.DataFrame(regression_filled, columns=X_test_missing.columns, index=X_test_missing.index)

    mice = IterativeImputer(random_state=0, max_iter=30, tol=1e-2).fit(X_train_clean.values)
    filled["MICE"] = pd.DataFrame(
        mice.transform(X_test_missing.values), columns=X_test_missing.columns, index=X_test_missing.index
    )
    return filled

### Steps 5-6: predict with the same model, compare RMSE across missing-ness levels

In [ ]:
rng = np.random.default_rng(42)
missing_ratios = [0.05, 0.10, 0.20, 0.30]
rmse_results = {}

for ratio in missing_ratios:
    X_test_missing = introduce_missing(wine_X_test, ratio, rng)
    filled_versions = impute_test_features(wine_X_train, X_test_missing)
    rmse_results[ratio] = {
        method: np.sqrt(mean_squared_error(wine_y_test, wine_rf.predict(X_filled)))
        for method, X_filled in filled_versions.items()
    }

rmse_df = pd.DataFrame(rmse_results).T
rmse_df.index.name = "missing_ratio"
print(f"Baseline (0% missing): {baseline_rmse:.4f}\n")
print(rmse_df.round(4))

In [ ]:
plt.figure(figsize=(7, 5))
for method in rmse_df.columns:
    plt.plot(rmse_df.index * 100, rmse_df[method], marker="o", label=method)
plt.axhline(baseline_rmse, color="gray", linestyle="--", label="baseline (0% missing)")
plt.xlabel("Missing Value Ratio (%)")
plt.ylabel("RMSE (quality score)")
plt.title("Prediction Error vs. Missing-Value Ratio, by Imputation Method")
plt.legend()
plt.show()

**Mode is consistently the worst** — collapsing every blank to a single
most common value throws away the real variation these continuous
measurements actually have. **Mean and median sit in the middle** and
degrade steadily as more values go missing. **Regression and MICE tend to
hold up best**, especially at the higher missing ratios (20-30%) — they use
the *other* measurements on the same row to make an informed guess instead
of a flat constant, so the more correlated the features are, the more that
pays off. The lesson from both sections together: the "safe-looking"
simple methods get noticeably worse as missingness increases, while the
model-based methods degrade more slowly — exactly why it's worth measuring,
not assuming.

Back to the Titanic pipeline — next we look at outliers.

### Finding outliers

Plotting `Fare` as a boxplot shows most passengers paid a modest fare, while
a handful paid a lot more. Of the methods mentioned in lecture (IQR,
Z-score, LOF, DBSCAN), we'll use the simplest: the **IQR method** — anything
outside `Q1 - 1.5*IQR` to `Q3 + 1.5*IQR` counts as an outlier.

In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot(titanic["Fare"])
plt.title("Fare Distribution (boxplot)")
plt.ylabel("Fare")
plt.show()

Q1, Q3 = titanic["Fare"].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outliers = titanic[(titanic["Fare"] < lower) | (titanic["Fare"] > upper)]

print(f"IQR range: [{lower:.2f}, {upper:.2f}]")
print(f"Passengers flagged as outliers: {len(outliers)} / {len(titanic)}")
outliers[["Pclass", "Sex", "Age", "Fare"]].sort_values("Fare", ascending=False).head(10)

**Remove them, or keep them?** As lecture emphasized, an outlier isn't
automatically an error. Here, the high-fare passengers are mostly genuine
**1st-class (Pclass=1) passengers** — not a measurement mistake, just people
who paid for an expensive ticket. In cases like this, it's common to either
**leave them as-is**, or, if a model turns out to be overly sensitive to
extreme values, **cap** them at some threshold (winsorizing) instead of
deleting rows outright.

### A rigorous comparison: does removing outliers actually help? (Wine Quality)

Same question as the missing-value sections above, but for outliers this
time: lecture covered several outlier-detection methods (IQR, Z-score,
LOF, PCA reconstruction error, Isolation Forest) — does it actually matter
*which* one you use before training a model? This reuses the Wine Quality
setup from before: contaminate the training data with about 5% synthetic
outliers, then compare the downstream `RandomForestRegressor` performance
with **no removal** against removing them with three different detectors:
**IQR**, **LOF** (Local Outlier Factor), and **PCA reconstruction error**.

Each synthetic outlier row is built by taking a real training row and
replacing its measurements, one column at a time, with that same column's
value from a different random wine — so every individual number is a
perfectly realistic reading (it really was measured on some wine), but the
*combination* on one row may no longer make chemical sense.

In [ ]:
def inject_multivariate_outliers(X, y, frac, rng):
    X = X.reset_index(drop=True).copy()
    y = y.reset_index(drop=True).copy()
    clean_X, clean_y = X.copy(), y.copy()
    n = len(X)
    n_outliers = int(frac * n)
    target_rows = rng.choice(n, size=n_outliers, replace=False)
    is_outlier = np.zeros(n, dtype=bool)
    is_outlier[target_rows] = True

    for row in target_rows:
        donor_rows = rng.integers(0, n, size=len(X.columns))
        for col, donor in zip(X.columns, donor_rows):
            X.loc[row, col] = clean_X.loc[donor, col]
        y.loc[row] = clean_y.loc[rng.integers(0, n)]
    return X, y, is_outlier


outlier_rng = np.random.default_rng(7)
contaminated_X, contaminated_y, is_true_outlier = inject_multivariate_outliers(
    wine_X_train, wine_y_train, frac=0.05, rng=outlier_rng
)
print(f"Injected {is_true_outlier.sum()} synthetic outliers into {len(contaminated_X)} training rows")

### Detecting the outliers three ways

- **IQR** — flag a row if *any single column* falls outside
  `Q1 - 1.5*IQR` to `Q3 + 1.5*IQR` for that column (same rule used on
  Titanic's `Fare` above, just applied to every column here).
- **LOF (Local Outlier Factor)** — compares each point's local density to
  its neighbors'; a point sitting in a much sparser neighborhood than its
  neighbors is flagged, regardless of which column makes it sparse.
- **PCA reconstruction error** — keep just enough principal components to
  explain most of the variance (5 components here, ~80%), reconstruct
  every row from those components, and flag the rows with the largest
  reconstruction error — i.e., the ones that don't fit the data's usual
  correlation structure.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.decomposition import PCA

iqr_flagged = np.zeros(len(contaminated_X), dtype=bool)
for col in wine_feature_cols:
    Q1, Q3 = contaminated_X[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    iqr_flagged |= (contaminated_X[col] < lower) | (contaminated_X[col] > upper)

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
lof_flagged = lof.fit_predict(contaminated_X.values) == -1

outlier_scaler = StandardScaler().fit(contaminated_X)
contaminated_X_scaled = outlier_scaler.transform(contaminated_X)
pca_detector = PCA(n_components=5, random_state=0).fit(contaminated_X_scaled)
reconstructed = pca_detector.inverse_transform(pca_detector.transform(contaminated_X_scaled))
reconstruction_error = np.sum((contaminated_X_scaled - reconstructed) ** 2, axis=1)
pca_flagged = reconstruction_error > np.quantile(reconstruction_error, 0.95)

detectors = {"IQR": iqr_flagged, "LOF": lof_flagged, "PCA reconstruction": pca_flagged}

# We know exactly which rows we injected, so we can directly check how many
# of them each detector actually caught.
n_true_outliers = is_true_outlier.sum()
print(f"{'method':20s} {'flagged':>8s} {f'caught (of {n_true_outliers} true)':>22s}")
for name, flagged in detectors.items():
    caught = (flagged & is_true_outlier).sum()
    print(f"{name:20s} {flagged.sum():8d} {caught:22d}")

### Does removing them actually improve the model?

In [ ]:
outlier_experiment_rmse = {}

rf_no_removal = RandomForestRegressor(n_estimators=200, random_state=42)
rf_no_removal.fit(contaminated_X, contaminated_y)
outlier_experiment_rmse["no removal"] = np.sqrt(
    mean_squared_error(wine_y_test, rf_no_removal.predict(wine_X_test))
)

for name, flagged in detectors.items():
    keep = ~flagged
    rf_cleaned = RandomForestRegressor(n_estimators=200, random_state=42)
    rf_cleaned.fit(contaminated_X[keep], contaminated_y[keep])
    outlier_experiment_rmse[name] = np.sqrt(
        mean_squared_error(wine_y_test, rf_cleaned.predict(wine_X_test))
    )

outlier_rmse_series = pd.Series(outlier_experiment_rmse)
print(outlier_rmse_series.round(4))
print(f"\n[for reference: baseline RMSE with no contamination at all: {baseline_rmse:.4f}]")

plt.figure(figsize=(7, 4))
outlier_rmse_series.plot(kind="barh", color="steelblue")
plt.axvline(baseline_rmse, color="gray", linestyle="--", label="no contamination at all")
plt.title("Test RMSE After Removing Outliers Three Different Ways")
plt.xlabel("RMSE (quality score)")
plt.legend()
plt.gca().invert_yaxis()
plt.show()

The numbers tell a more nuanced story than "always remove outliers."
**IQR ends up flagging far more rows than it catches true outliers** —
because it checks one column at a time, a row only needs *one* unusually
large or small measurement to get flagged, so it sweeps up a lot of
perfectly ordinary wines along with the real contamination. Removing all
of those hurts more than it helps: IQR-cleaned training data can produce a
*worse* model than not removing anything at all. **LOF and PCA
reconstruction error**, which both judge a row by its *entire* combination
of measurements at once, flag a much more tightly-targeted set of rows and
genuinely bring the model closer to the uncontaminated baseline. The
takeaway: an outlier detector that only ever looks at one column at a time
has a real blind spot for rows that are only unusual *in combination* —
which is exactly the kind of outlier a method like LOF or PCA is built to
catch.

### Encoding categorical columns: ordinal vs. nominal

Applying the distinction from lecture to this dataset:

- **`Pclass`** (1st < 2nd < 3rd class) is an **ordinal** category — it's
  already numeric, and that order is meaningful, so we can **use it as-is**.
- **`Sex`** and **`Embarked`** are **nominal** categories with no order, so
  we use **one-hot encoding** like in the earlier section.

In [ ]:
cat_encoded = pd.get_dummies(titanic[["Sex", "Embarked"]])
numeric_cols = ["Age", "Fare", "SibSp", "Parch"]

X = pd.concat([titanic[["Pclass"] + numeric_cols], cat_encoded], axis=1)
y = titanic["Survived"]
X.head()

### Splitting into train / validation / test, without leaking into the scaler

Two rules from lecture, applied here:

1. **Three-way split**: divide the data into train/validation/test (a common
   ratio is 6:2:2 or 7:2:1). Validation is used during training to pick the
   best model and tune hyperparameters; test is touched exactly once, at the
   very end.
2. **Scale using training data only**: when you `fit` a `StandardScaler`, it
   must only see the **training** data. Fitting it on the full dataset (or
   anything that includes val/test) leaks a bit of "future" information into
   the mean/std it learns — that's **data leakage**.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)
print(f"train/val/test sizes: {len(X_train)} / {len(X_val)} / {len(X_test)}")
print(f"ratios: {len(X_train)/len(X):.2f} / {len(X_val)/len(X):.2f} / {len(X_test)/len(X):.2f}")

In [ ]:
# Wrong way: fit the scaler on the full dataset (val/test information leaks in)
scaler_wrong = StandardScaler().fit(X[numeric_cols])

# Right way: fit only on train, then just transform() val/test
scaler_right = StandardScaler().fit(X_train[numeric_cols])

print(f"Age mean/std, fit on FULL data (leaky):   {scaler_wrong.mean_[0]:.3f} / {scaler_wrong.scale_[0]:.3f}")
print(f"Age mean/std, fit on TRAIN only (correct): {scaler_right.mean_[0]:.3f} / {scaler_right.scale_[0]:.3f}")

X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler_right.transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler_right.transform(X_val[numeric_cols])
X_test_scaled[numeric_cols] = scaler_right.transform(X_test[numeric_cols])

print("\ntrain mean after scaling (should be ~0):", X_train_scaled[numeric_cols].mean().mean().round(6))

Notice the two mean/std values are (slightly) different. This dataset is
small enough that the difference is minor, but with a larger dataset, or
train/val/test sets with different distributions, that gap can actually
affect model performance — and more fundamentally, it breaks the premise
that val/test are supposed to look like "data the model hasn't seen yet."

**To recap:** handling missing values appropriately per column, flagging
outliers and deciding whether to keep or cap them, encoding ordinal vs.
nominal categories correctly, and fitting the scaler on training data
only — all of this is the **Data Preprocessing** stage from the ML
pipeline at the start of lecture. **Part 2**, right below, uses these same validation/test splits to
actually compute metrics like the confusion matrix, precision, and
recall.

## Part 2: Model Evaluation — Breast Cancer Diagnosis Evaluator

**Theme:** Model evaluation — cross-validation & evaluation metrics

So far we've measured models with a single train/test split and a single
accuracy number. That's not enough for a real decision — especially in a
medical context, where the *kind* of mistake matters a lot: missing a
malignant tumor (false negative) is far worse than an unnecessary follow-up
test (false positive).

**Dataset:** scikit-learn's built-in breast cancer dataset — 569 patients,
30 measurements from a tumor scan, label = malignant or benign.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_curve, roc_auc_score, precision_recall_curve,
)

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
print("Classes:", dict(zip(data.target_names, [0, 1])))
print("Shape:", X.shape)
print("Class balance:", np.bincount(y), "(0=malignant, 1=benign)")

## 1. A single train/test split — and why it's not enough

We build a `Pipeline` so scaling always happens *inside* the same fit/predict
step (never fit the scaler on test data — that would leak information).

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1, stratify=y
)
model.fit(X_train, y_train)
single_split_acc = accuracy_score(y_test, model.predict(X_test))
print(f"Accuracy on THIS ONE split: {single_split_acc:.2%}")

In [ ]:
# What if we'd used a different random split? Try several seeds.
accs = []
for seed in range(10):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
    accs.append(accuracy_score(yte, m.predict(Xte)))

print("Accuracy across 10 different random splits:", [f"{a:.2%}" for a in accs])
print(f"Range: {min(accs):.2%} to {max(accs):.2%} — a single split can be lucky or unlucky!")

## 2. k-fold cross-validation: a more honest estimate

Instead of one split, **k-fold cross-validation** splits the data into `k`
parts, trains on `k-1` of them and tests on the last, and rotates — so every
example gets used for testing exactly once. We report the average.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

print("5-fold CV accuracy per fold:", [f"{s:.2%}" for s in cv_scores])
print(f"Mean: {cv_scores.mean():.2%}  (+/- {cv_scores.std():.2%})")

## 3. Confusion matrix: what kind of mistakes does it make?

Accuracy hides *which* mistakes happen. The confusion matrix doesn't.

In [ ]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=data.target_names)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_test, y_pred, target_names=data.target_names))

## 4. Precision, recall, and why accuracy alone can mislead

- **Precision** — of everything we predicted "benign", how much really was benign?
- **Recall** — of everything that really was "malignant", how much did we catch?

In this task, missing a malignant tumor (a false negative for the malignant
class) is the costly mistake — so **recall on the malignant class** matters a
lot, sometimes more than overall accuracy.

In [ ]:
# malignant is class 0 in this dataset — let's look at recall for it specifically
report = classification_report(y_test, y_pred, target_names=data.target_names, output_dict=True)
print(f"Recall on malignant cases: {report['malignant']['recall']:.2%}")
print(f"Precision on malignant cases: {report['malignant']['precision']:.2%}")

## 5. ROC curve and AUC

The model doesn't just output a class — it outputs a *probability*. The ROC
curve shows the tradeoff between catching more true positives (recall) and
generating more false alarms, as we sweep the decision threshold.

In [ ]:
y_scores = model.predict_proba(X_test)[:, 1]  # probability of "benign"
fpr, tpr, thresholds = roc_curve(y_test, y_scores)
auc = roc_auc_score(y_test, y_scores)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="random guessing")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve")
plt.legend()
plt.show()